# Topic: SQL Missing Values Detection & NULL Handling

## Definition (30-second explanation)
*   In SQL, `NULL` represents a missing, unknown, or inapplicable value. 
*   Handling missing values involves auditing tables for `NULL` rates and safely transforming or ignoring them using functions like `COALESCE()`, `NULLIF()`, and `IS NULL` to prevent inaccurate calculations or pipeline failures.

## Why Interviewers Ask This
*   **Counterintuitive Logic:** `NULL` behavior (Three-Valued Logic: True, False, Unknown) frequently traps beginners. Interviewers want to ensure you know that `NULL != NULL`.
*   **Data Reliability:** In ML and Data Science, feeding unhandled `NULL` values into a model or a dashboard causes silent failures or skewed metrics.
*   **Defensive Coding:** Tests your ability to write robust code that anticipates edge cases (like division by zero).

## Core Concepts
*   **Three-Valued Logic:** Comparisons with `NULL` evaluate to `UNKNOWN` (not True, not False).
*   **IS NULL / IS NOT NULL:** The *only* correct operators to check for missing values.
*   **COALESCE(val1, val2, ...):** Returns the first non-`NULL` value in a list.
*   **NULLIF(val1, val2):** Returns `NULL` if `val1 == val2`; otherwise returns `val1`. Essential for preventing division by zero errors.

## When to Use
*   **Auditing Data Quality:** Calculating the percentage of missing features before training an ML model.
*   **ETL Pipelines:** Providing default fallback values (e.g., replacing a `NULL` phone number with 'No Contact Info').
*   **Aggregations:** Ensuring averages or counts are calculated over the correct denominator.

## Advantages
*   **COALESCE:** Much cleaner and more readable than writing nested `CASE WHEN` statements for simple defaults.
*   **NULLIF:** Provides a mathematically safe way to handle denominator zeroes without filtering out rows entirely.

## Limitations
*   `COALESCE` requires all arguments to be of the same (or implicitly convertible) data type.
*   Overusing `COALESCE` to mask poor data quality upstream can hide systemic data engineering bugs.

## Common Comparisons
*   **COUNT(*) vs. COUNT(column):** `COUNT(*)` counts *all* rows, including those with `NULL`s. `COUNT(column)` ignores rows where that specific column is `NULL`.
*   **COALESCE() vs. IFNULL():** `COALESCE` is ANSI SQL standard and accepts multiple arguments. `IFNULL` is specific to certain dialects (like MySQL) and only takes two arguments.

## Common Interview Traps
*   **The Equality Trap:** Writing `WHERE column = NULL`. This always evaluates to `UNKNOWN` and returns zero rows.
*   **The JOIN Trap:** Joining on a column that contains `NULL`s. `NULL` does not match with `NULL` in a standard `JOIN`, dropping those records.
*   **The Math Trap:** Any arithmetic operation with a `NULL` (e.g., `revenue + NULL`) results in `NULL`.
*   **Aggregate Skew:** Forgetting that `SUM()` and `AVG()` silently ignore `NULL` values, which might artificially inflate your average.

## Python / SQL Syntax
```sql
-- 1. Auditing NULL Percentage
SELECT 
    COUNT(*) AS total_rows,
    ROUND(100.0 * SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_null_email
FROM users;

-- 2. Safe Division & Fallbacks
SELECT 
    user_id,
    COALESCE(phone, 'No Phone') AS safe_phone,
    -- NULLIF prevents division by zero. COALESCE defaults the final result to 0 if NULL.
    COALESCE(total_sales / NULLIF(total_visits, 0), 0) AS conversion_rate
FROM user_metrics;
```

## 45-Second Interview Answer

"In SQL, handling missing data requires understanding that NULL means 'unknown', not zero or empty. Because of three-valued logic, we must use IS NULL instead of standard equality operators. In my data prep pipelines for machine learning, I regularly audit NULL percentages using conditional aggregation. To handle them, I use COALESCE to provide safe default values and NULLIF to prevent division-by-zero errors when calculating ratios, ensuring the downstream models receive robust, predictable data."

## Practice Questions:

### Q1: Defensive Feature Engineering (COALESCE & NULLIF)
**Mock Schema and Data:**
```sql
CREATE TABLE customers_mock (
    customer_id INT PRIMARY KEY,
    country VARCHAR(50),
    total_revenue DECIMAL(10,2),
    total_orders INT
);

INSERT INTO customers_mock VALUES 
(1, 'USA', 500.00, 5),
(2, NULL, 0.00, 0),       -- No country, zero orders
(3, 'UK', NULL, 2),       -- Missing revenue data
(4, 'India', 150.00, 0);  -- Edge case: Revenue exists, but orders are 0 (data anomaly)
```

**Context:** 
You are building a feature table for a machine learning model. You need to calculate a safe "Average Order Value" (AOV) from `total_revenue` and `total_orders`, and fill missing geographic data.

**Question:** 
Write a query to return `customer_id`, `clean_country` (default 'Unknown'), and `safe_aov`. You must prevent division-by-zero, and default the final AOV to `0.00` if the result is NULL.

**Answer:**
```sql
SELECT 
    customer_id,
    COALESCE(country, 'Unknown') AS clean_country,
    COALESCE(total_revenue / NULLIF(total_orders, 0), 0.00) AS safe_aov
FROM customers_mock;
```

**Interview Tips:**

**Nested Functions:** Do not be afraid to nest NULLIF inside COALESCE. It is the industry standard for safe division.

**Order of Operations:** The NULLIF evaluates first. If total_orders is 0, it becomes NULL. Any number divided by NULL is NULL. Finally, the outer COALESCE catches that NULL (or a NULL total_revenue) and turns it into 0.00.

### Q2:The Aggregate Trap (COUNT vs COUNT)

**Context:** 
You are profiling data to understand delivery completion rates by looking at the `shipped_date` column, which contains NULLs for unshipped orders.

**Question:** 
1. What is the difference between `COUNT(*)` and `COUNT(shipped_date)`?
2. Write a single SQL statement to calculate the percentage of shipments that are *missing* a `shipped_date`.

**Ideal Interview Answer:**
1. `COUNT(*)` counts every single row in the table, regardless of what data it contains. `COUNT(shipped_date)` counts only the rows where `shipped_date` is strictly NOT NULL. 

2. 
```sql
-- Method 1: Mathematical Subtraction (Fastest)
SELECT 
    ROUND(100.0 * (COUNT(*) - COUNT(shipped_date)) / COUNT(*), 2) AS pct_missing
FROM shipments;

-- Method 2: Conditional Aggregation (Most flexible for multiple columns)
SELECT 
    ROUND(100.0 * SUM(CASE WHEN shipped_date IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing
FROM shipments;
```

**Interview Tips:**

**Trap:** Differentiate between calculating the "fill rate" (present values) versus the "missing rate" (null values).

**Integer Division:** Always multiply by 100.0 (with the decimal) before dividing. If you multiply by 100 or divide first, many SQL engines (like PostgreSQL and SQL Server) will perform integer division, resulting in 0.